# 03 — Data Cleaning Pipeline

**Parkinson's Keystroke Project — TAPPY Mendeley v3**

This notebook implements the full label-free cleaning pipeline.

**Outputs:**
- `data/processed/features_clean.csv` — timing features + `parkinsons_label` only (X + y)
- `data/processed/participant_info.csv` — clinical fields for description / error analysis only
- `data/processed/train_ids.csv` / `test_ids.csv` — stratified 80/20 split participant IDs

**Rules:**
1. No clinical leakage fields (UPDRS, Tremors, Levodopa, DA, MAOB, Other, DiagnosisYear, Sided, Impact) in X.
2. Cleaning is label-free; split happens after features_clean.csv is saved.
3. All label-aware EDA uses train data only.

---

## 0. Setup

In [ ]:
import re
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

LOG_DIR      = PROJECT_ROOT / 'data' / 'raw' / 'keystroke_logs'
META_DIR     = PROJECT_ROOT / 'data' / 'raw' / 'participant_metadata'
PROCESSED    = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

FEATURES_OUT      = PROCESSED / 'features_clean.csv'
PARTICIPANT_INFO   = PROCESSED / 'participant_info.csv'
TRAIN_IDS_OUT     = PROCESSED / 'train_ids.csv'
TEST_IDS_OUT      = PROCESSED / 'test_ids.csv'

# ── Constants ─────────────────────────────────────────────────────────────────
VALID_HANDS      = {'L', 'R', 'S'}
VALID_DIRECTIONS = {'LL', 'LR', 'RL', 'RR', 'LS', 'SL', 'RS', 'SR', 'SS'}
MIN_KEYSTROKES   = 1000   # minimum valid keystrokes per participant (justify from histogram below)
HOLD_UPPER_MS    = None   # set after EDA histogram in Section 3 — leave None to auto-set at 99.9th pct
RANDOM_STATE     = 42
TEST_SIZE        = 0.20

# ── Filename regexes ─────────────────────────────────────────────────────────
LOG_RE  = re.compile(r'^(?P<pid>[A-Za-z0-9]+)_(?P<ym>\d{4})\.txt$', re.IGNORECASE)
META_RE = re.compile(r'^User_(?P<pid>[A-Za-z0-9]+)\.txt$',            re.IGNORECASE)

# ── Log columns (9 cols: 8 real + trailing tab → empty 9th) ──────────────────
LOG_COLS = ['participant_id', 'date', 'time', 'hand', 'hold_ms',
            'direction', 'latency1_ms', 'latency2_ms', '_trailing']

print('Setup complete.')
print(f'Keystroke log dir  : {LOG_DIR}')
print(f'Metadata dir       : {META_DIR}')
print(f'Output dir         : {PROCESSED}')

---
## 1. Parse Participant Metadata

- Convert boolean fields to 0/1.
- **Leakage fields** (UPDRS, Tremors, Levodopa, DA, MAOB, Other, DiagnosisYear, Sided, Impact) are stored separately in `participant_info.csv` and **not included in X**.
- Only `parkinsons_label`, `birth_year`, and `gender` are retained in the main table, and even those are for confound checking — not features.

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────
def normalize_key(raw: str) -> str:
    """'MAOB ' -> 'maob'  |  'BirthYear' -> 'birth_year' (snake_case)"""
    s = re.sub(r'(?<=[a-z])(?=[A-Z])', '_', raw.strip())
    s = re.sub(r'[^a-z0-9]+', '_', s.lower())
    return s.strip('_')


def parse_bool(val):
    """'True'/'False'/'Yes'/'No' -> 1/0; anything else -> pd.NA"""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return pd.NA
    s = str(val).strip().lower()
    if s in {'true', '1', 'yes', 'y'}:  return 1
    if s in {'false', '0', 'no', 'n'}:  return 0
    return pd.NA


def parse_meta_file(path: Path) -> dict:
    """Parse one User_*.txt metadata file into a flat dict."""
    record = {}
    try:
        with path.open('r', encoding='utf-8-sig', errors='replace') as f:
            for line in f:
                if ':' not in line or not line.strip():
                    continue
                key_raw, _, val_raw = line.partition(':')
                record[normalize_key(key_raw)] = val_raw.strip()
    except OSError as e:
        log.warning(f'Cannot read {path.name}: {e}')
    return record


# ── Parse all metadata files ──────────────────────────────────────────────────
meta_files = sorted(p for p in META_DIR.iterdir()
                    if p.is_file() and META_RE.match(p.name))
print(f'Metadata files found: {len(meta_files)}')

rows = []
for path in meta_files:
    pid = META_RE.match(path.name).group('pid').upper()
    rec = parse_meta_file(path)
    rec['participant_id'] = pid
    rows.append(rec)

meta_raw = pd.DataFrame(rows)

# ── Normalise Parkinsons label ────────────────────────────────────────────────
if 'parkinsons' not in meta_raw.columns:
    meta_raw['parkinsons'] = pd.NA

meta_raw['parkinsons_label'] = (
    meta_raw['parkinsons'].apply(parse_bool).astype('Int8')  # 0/1 or NA
)

print(f'Total metadata records         : {len(meta_raw)}')
print(f'  Valid parkinsons labels       : {meta_raw["parkinsons_label"].notna().sum()}')
print(f'  Missing/invalid labels        : {meta_raw["parkinsons_label"].isna().sum()}')
print(f'  Duplicate participant_id rows : {meta_raw.duplicated("participant_id").sum()}')

In [ ]:
# ── Separate label + safe diagnostics from leakage fields ────────────────────

# Fields that would leak the label (only PD patients have them populated)
LEAKAGE_FIELDS = [
    'updrs', 'tremors', 'levodopa', 'da', 'maob', 'other',
    'diagnosis_year', 'sided', 'impact'
]

# Fields safe to keep for confound checks (but NOT in X)
SAFE_DIAGNOSTICS = ['birth_year', 'gender']

# Build participant_info table (never used as ML features)
info_cols = ['participant_id', 'parkinsons_label'] + SAFE_DIAGNOSTICS
info_cols += [c for c in LEAKAGE_FIELDS if c in meta_raw.columns]

participant_info = meta_raw[[c for c in info_cols if c in meta_raw.columns]].copy()
participant_info = participant_info.drop_duplicates('participant_id', keep='first')

# Build label-only table for joining to features
meta_clean = (
    meta_raw[['participant_id', 'parkinsons_label']]
    .drop_duplicates('participant_id', keep='first')
    .reset_index(drop=True)
)

print(f'participant_info shape : {participant_info.shape}')
print(f'meta_clean (label-only): {meta_clean.shape}')
print()
print('Label distribution in metadata:')
print(meta_clean['parkinsons_label'].value_counts(dropna=False).to_string())

---
## 2. Parse Keystroke Log Files

Files are tab-separated, 9 columns (8 data + trailing tab), CRLF line endings.  
Empty files (size = 0) are skipped. Malformed lines are skipped via `on_bad_lines='skip'`.

**Resolve file count:** If the Mendeley v3 dataset should have 2,166 files, check whether `Archived Users.zip` in `keystroke_logs/` contains the missing file — extract it first if so.

In [ ]:
# ── Discover valid log files ──────────────────────────────────────────────────
all_log_paths = sorted(p for p in LOG_DIR.iterdir()
                       if p.is_file() and LOG_RE.match(p.name))
empty_paths   = [p for p in all_log_paths if p.stat().st_size == 0]
valid_paths   = [p for p in all_log_paths if p.stat().st_size > 0]

print(f'Total log files matching pattern : {len(all_log_paths)}')
print(f'  Empty (skipped)                : {len(empty_paths)}')
print(f'  Non-empty (to parse)           : {len(valid_paths)}')
print()
if empty_paths:
    print('Empty files skipped:')
    for p in empty_paths:
        print(f'  {p.name}')

In [ ]:
# ── Parse all non-empty log files ─────────────────────────────────────────────
# NOTE: This pools ALL rows across all session files per participant.
# Each file contributes equally at the row level, so large files dominate
# (which is correct — they represent more typing activity).

chunks = []
parse_errors = 0

for path in valid_paths:
    try:
        chunk = pd.read_csv(
            path,
            sep='\t',
            header=None,
            names=LOG_COLS,
            dtype=str,
            on_bad_lines='skip',   # pandas >= 1.3 replacement for error_bad_lines
            engine='python',
        )
        chunks.append(chunk)
    except Exception as e:
        log.warning(f'Failed to parse {path.name}: {e}')
        parse_errors += 1

raw_logs = pd.concat(chunks, ignore_index=True)
del chunks

print(f'Files parsed        : {len(valid_paths) - parse_errors}  (errors: {parse_errors})')
print(f'Total raw rows      : {len(raw_logs):,}')
print()
print('Column dtypes after read:')
print(raw_logs.dtypes)

---
## 3. Validate Codes & Apply Range Rules

Each removal rule is logged separately so we can track the effect of each filter.

In [ ]:
# ── Normalise column types ────────────────────────────────────────────────────
df = raw_logs.copy()

# Strip whitespace from string columns
for col in ['participant_id', 'hand', 'direction']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.upper()

# Convert timing columns to numeric (coerce bad values -> NaN)
for col in ['hold_ms', 'latency1_ms', 'latency2_ms']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop the trailing empty column
df.drop(columns=['_trailing'], inplace=True, errors='ignore')

n_start = len(df)
removal_log = {'start': n_start}
print(f'Rows before filtering: {n_start:,}')

In [ ]:
# ── Rule 1: Validate participant_id (must not be null / numeric) ─────────────
mask_bad_pid = df['participant_id'].isna() | df['participant_id'].str.match(r'^\d+$').fillna(False)
n_bad_pid = mask_bad_pid.sum()
df = df[~mask_bad_pid].copy()
removal_log['invalid_participant_id'] = n_bad_pid
print(f'Removed (invalid participant_id): {n_bad_pid:,}  |  Remaining: {len(df):,}')

In [ ]:
# ── Rule 2: Validate hand in {L, R, S} ────────────────────────────────────────
# Shifted rows often have date strings like '160722' in this column.
mask_bad_hand = ~df['hand'].isin(VALID_HANDS)
n_bad_hand = mask_bad_hand.sum()
df = df[~mask_bad_hand].copy()
removal_log['invalid_hand'] = n_bad_hand
print(f'Removed (invalid hand)          : {n_bad_hand:,}  |  Remaining: {len(df):,}')

In [ ]:
# ── Rule 3: Validate direction in valid set ───────────────────────────────────
mask_bad_dir = ~df['direction'].isin(VALID_DIRECTIONS)
n_bad_dir = mask_bad_dir.sum()
df = df[~mask_bad_dir].copy()
removal_log['invalid_direction'] = n_bad_dir
print(f'Removed (invalid direction)     : {n_bad_dir:,}  |  Remaining: {len(df):,}')

In [ ]:
# ── Rule 4: hold_ms coercion NaN (unparseable string after earlier filters) ──
mask_null_hold = df['hold_ms'].isna()
n_null_hold = mask_null_hold.sum()
df = df[~mask_null_hold].copy()
removal_log['null_hold_ms'] = n_null_hold
print(f'Removed (null hold_ms)          : {n_null_hold:,}  |  Remaining: {len(df):,}')

In [ ]:
# ── Rule 5: hold_ms <= 0 ─────────────────────────────────────────────────────
mask_neg_hold = df['hold_ms'] <= 0
n_neg_hold = mask_neg_hold.sum()
df = df[~mask_neg_hold].copy()
removal_log['nonpositive_hold_ms'] = n_neg_hold
print(f'Removed (hold_ms <= 0)          : {n_neg_hold:,}  |  Remaining: {len(df):,}')

In [ ]:
# ── EDA: hold_ms distribution — choose upper bound ───────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full range histogram
axes[0].hist(df['hold_ms'].clip(0, 2000), bins=200, color='steelblue', edgecolor='none')
axes[0].set_title('hold_ms distribution (clipped at 2000 ms)')
axes[0].set_xlabel('hold_ms')
axes[0].set_ylabel('count')

# Upper tail
tail = df.loc[df['hold_ms'] > 500, 'hold_ms']
axes[1].hist(tail.clip(0, 5000), bins=100, color='coral', edgecolor='none')
axes[1].set_title('hold_ms > 500 ms (upper tail)')
axes[1].set_xlabel('hold_ms')

plt.tight_layout()
plt.show()

pcts = df['hold_ms'].quantile([0.90, 0.95, 0.99, 0.999, 1.0])
print('hold_ms percentiles:')
print(pcts.to_string())

print()
print('>>> Set HOLD_UPPER_MS in the Setup cell based on the histogram above.')
print('    Recommendation: inspect where the bulk density ends; 99.9th pct is a safe default.')

In [ ]:
# ── Rule 6: hold_ms upper bound (auto-set to 99.9th pct if not manually set) ─
if HOLD_UPPER_MS is None:
    HOLD_UPPER_MS = float(df['hold_ms'].quantile(0.999))
    print(f'HOLD_UPPER_MS auto-set to 99.9th pct: {HOLD_UPPER_MS:.1f} ms')

mask_high_hold = df['hold_ms'] > HOLD_UPPER_MS
n_high_hold = mask_high_hold.sum()
df = df[~mask_high_hold].copy()
removal_log['hold_ms_above_upper'] = n_high_hold
print(f'Removed (hold_ms > {HOLD_UPPER_MS:.0f} ms)    : {n_high_hold:,}  |  Remaining: {len(df):,}')

In [ ]:
# ── Rule 7: latency1_ms and latency2_ms <= 0 -> NaN (not row-drop) ──────────
# Negative latency can occur when the next key is pressed before the previous
# is released (overlapping keystrokes). These are set to NaN so per-participant
# stats use only meaningful observations, but the row is kept (hold_ms is valid).
n_neg_lat1 = (df['latency1_ms'] <= 0).sum()
n_neg_lat2 = (df['latency2_ms'] <= 0).sum()
df.loc[df['latency1_ms'] <= 0, 'latency1_ms'] = np.nan
df.loc[df['latency2_ms'] <= 0, 'latency2_ms'] = np.nan
removal_log['latency1_set_nan'] = n_neg_lat1
removal_log['latency2_set_nan'] = n_neg_lat2
print(f'latency1_ms <= 0 set to NaN     : {n_neg_lat1:,}')
print(f'latency2_ms <= 0 set to NaN     : {n_neg_lat2:,}')
print(f'Rows remaining                  : {len(df):,}')

In [ ]:
# ── Removal summary ───────────────────────────────────────────────────────────
print('=== Keystroke Cleaning Summary ===')
total_removed = removal_log['start'] - len(df)
for rule, count in removal_log.items():
    if rule == 'start':
        print(f'  Starting rows          : {count:>10,}')
    else:
        print(f'  {rule:<30}: {count:>10,}')
print(f'  Final rows             : {len(df):>10,}')
print(f'  Total removed          : {total_removed:>10,}  ({100*total_removed/removal_log["start"]:.2f}%)')

---
## 4. EDA: Minimum Keystroke Threshold

Inspect the per-participant valid-keystroke distribution to justify `MIN_KEYSTROKES = 1000`.
This is done on all participants (label-free) — do NOT split by label here.

In [ ]:
keystrokes_per_participant = df.groupby('participant_id').size().rename('n_valid_keys')

print('Per-participant valid keystroke count distribution:')
print(keystrokes_per_participant.describe(percentiles=[.05,.10,.25,.50,.75,.90,.95,.99]).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
keystrokes_per_participant.hist(bins=50, ax=axes[0], color='steelblue', edgecolor='none')
axes[0].axvline(MIN_KEYSTROKES, color='red', linewidth=2, label=f'MIN_KEYSTROKES={MIN_KEYSTROKES}')
axes[0].set_title('Keystrokes per participant')
axes[0].set_xlabel('n_valid_keys')
axes[0].legend()

keystrokes_per_participant.clip(upper=20000).hist(bins=50, ax=axes[1], color='steelblue', edgecolor='none')
axes[1].axvline(MIN_KEYSTROKES, color='red', linewidth=2)
axes[1].set_title('Keystrokes per participant (clipped at 20k)')
axes[1].set_xlabel('n_valid_keys')
plt.tight_layout()
plt.show()

n_below = (keystrokes_per_participant < MIN_KEYSTROKES).sum()
print(f'\nParticipants with < {MIN_KEYSTROKES} valid keystrokes: {n_below}')
print(f'>>> If this seems high, consider lowering MIN_KEYSTROKES and re-running.')

---
## 5. Compute Per-Participant Features

Pool ALL valid keystrokes per participant, compute stats once.  
Stats: **mean, std, median, skew, kurtosis, min, max, IQR, CV** for each timing dimension.

- **hold_ms**: overall + by hand (L, R, S) + L-R diff and ratio
- **latency1_ms** (inter-key onset): overall + by direction
- **latency2_ms** (flight / release-to-press): overall + by direction

In [ ]:
def timing_stats(series: pd.Series, prefix: str) -> dict:
    """Compute 9 descriptive stats for a timing series."""
    s = series.dropna()
    if len(s) < 2:
        return {f'{prefix}_{k}': np.nan for k in
                ['mean','std','median','skew','kurt','min','max','iqr','cv']}
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    mn = s.mean()
    return {
        f'{prefix}_mean'   : mn,
        f'{prefix}_std'    : s.std(),
        f'{prefix}_median' : s.median(),
        f'{prefix}_skew'   : float(scipy_stats.skew(s)),
        f'{prefix}_kurt'   : float(scipy_stats.kurtosis(s)),
        f'{prefix}_min'    : s.min(),
        f'{prefix}_max'    : s.max(),
        f'{prefix}_iqr'    : q3 - q1,
        f'{prefix}_cv'     : s.std() / mn if mn != 0 else np.nan,
    }


def compute_participant_features(grp: pd.DataFrame) -> dict:
    feats = {}

    # ── Overall timing stats ─────────────────────────────────────────────────
    feats.update(timing_stats(grp['hold_ms'],     'hold'))
    feats.update(timing_stats(grp['latency1_ms'], 'lat1'))
    feats.update(timing_stats(grp['latency2_ms'], 'lat2'))

    # ── hold_ms by hand ───────────────────────────────────────────────────────
    for hand in ['L', 'R', 'S']:
        subset = grp.loc[grp['hand'] == hand, 'hold_ms']
        feats.update(timing_stats(subset, f'hold_{hand}'))

    # ── L-R hold difference and ratio ────────────────────────────────────────
    hold_L = feats.get('hold_L_mean', np.nan)
    hold_R = feats.get('hold_R_mean', np.nan)
    feats['hold_LR_diff']  = hold_L - hold_R if not (np.isnan(hold_L) or np.isnan(hold_R)) else np.nan
    feats['hold_LR_ratio'] = (hold_L / hold_R
                               if not (np.isnan(hold_L) or np.isnan(hold_R) or hold_R == 0)
                               else np.nan)

    # ── latency1 and latency2 by direction ────────────────────────────────────
    for direction in VALID_DIRECTIONS:
        subset = grp[grp['direction'] == direction]
        feats.update(timing_stats(subset['latency1_ms'], f'lat1_{direction}'))
        feats.update(timing_stats(subset['latency2_ms'], f'lat2_{direction}'))

    # ── Diagnostics (NOT in X) ────────────────────────────────────────────────
    feats['n_keystrokes'] = len(grp)

    return feats


print('Computing per-participant features (this may take a few minutes)...')

feature_rows = []
for pid, grp in df.groupby('participant_id'):
    row = compute_participant_features(grp)
    row['participant_id'] = pid
    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)
print(f'Feature matrix shape (before join): {features_df.shape}')
print(f'  Participants: {features_df["participant_id"].nunique()}')
features_df.head(3)

---
## 6. Join, Filter, Assert

In [ ]:
# ── Inner join: drops 145 log-only + 37 metadata-only participants ────────────
joined = features_df.merge(meta_clean, on='participant_id', how='inner')
n_after_join = len(joined)
print(f'After inner join                 : {n_after_join}  (dropped {len(features_df) - n_after_join} log-only + metadata-only)')

In [ ]:
# ── Explicit dropna on parkinsons_label (inner join does NOT do this) ─────────
n_before_label_drop = len(joined)
joined = joined.dropna(subset=['parkinsons_label']).copy()
n_dropped_label = n_before_label_drop - len(joined)
print(f'Dropped (invalid parkinsons_label): {n_dropped_label}')
print(f'After label filter               : {len(joined)}')

# Assert: must be <= 606 (the 606 comes from the collaborator EDA)
assert len(joined) <= 606, f'Expected <= 606 but got {len(joined)}'
print(f'Assert <= 606 passed ✓')

In [ ]:
# ── Minimum keystroke threshold ───────────────────────────────────────────────
n_before_thresh = len(joined)
joined = joined[joined['n_keystrokes'] >= MIN_KEYSTROKES].copy()
n_dropped_thresh = n_before_thresh - len(joined)
print(f'Dropped (< {MIN_KEYSTROKES} keystrokes)      : {n_dropped_thresh}')
print(f'After keystroke threshold        : {len(joined)}')

print()
print('Final label distribution:')
print(joined['parkinsons_label'].value_counts().to_string())

In [ ]:
# ── Confirm no zero-valid-row participants slipped through ────────────────────
assert (joined['n_keystrokes'] >= MIN_KEYSTROKES).all(), 'Some participants below min keystrokes!'
assert joined['parkinsons_label'].notna().all(), 'Null labels present in final dataset!'
assert joined['participant_id'].nunique() == len(joined), 'Duplicate participant IDs!'
print(f'All assertions passed ✓')
print(f'Final dataset shape: {joined.shape}')

---
## 7. Save Outputs

In [ ]:
# ── Feature columns: timing stats + label only (NO n_keystrokes, BirthYear, etc. in X) ──
DIAGNOSTIC_COLS = ['participant_id', 'n_keystrokes']  # excluded from X, kept for reference

feature_cols = [c for c in joined.columns
                if c not in DIAGNOSTIC_COLS and c != 'parkinsons_label']

# features_clean.csv: timing features + label
features_clean = joined[['participant_id', 'parkinsons_label'] + feature_cols].copy()
features_clean.to_csv(FEATURES_OUT, index=False)
print(f'Saved features_clean.csv  -> {FEATURES_OUT}')
print(f'  Shape: {features_clean.shape}')
print(f'  Feature columns: {len(feature_cols)}')

# participant_info.csv: clinical / diagnostic fields (NOT used in ML)
info_out = joined[['participant_id', 'n_keystrokes']].merge(
    participant_info, on='participant_id', how='left'
)
info_out.to_csv(PARTICIPANT_INFO, index=False)
print(f'Saved participant_info.csv -> {PARTICIPANT_INFO}')
print(f'  Shape: {info_out.shape}')

---
## 8. Stratified Train / Test Split

- Participant-level stratified 80/20 split on `parkinsons_label`.
- Split IDs are saved. **All subsequent label-aware EDA uses train data only.**
- Feature selection, scaling, imputation, PCA, SMOTE belong inside sklearn Pipelines in CV — **not here**.

In [ ]:
from sklearn.model_selection import train_test_split

ids   = features_clean['participant_id'].values
y_all = features_clean['parkinsons_label'].astype(int).values

ids_train, ids_test, _, _ = train_test_split(
    ids, y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE
)

pd.DataFrame({'participant_id': ids_train}).to_csv(TRAIN_IDS_OUT, index=False)
pd.DataFrame({'participant_id': ids_test }).to_csv(TEST_IDS_OUT,  index=False)

df_train = features_clean[features_clean['participant_id'].isin(ids_train)].copy()
df_test  = features_clean[features_clean['participant_id'].isin(ids_test) ].copy()

print(f'Train: {len(df_train)} participants  |  PD={df_train["parkinsons_label"].sum()}, Healthy={(df_train["parkinsons_label"]==0).sum()}')
print(f'Test : {len(df_test)}  participants  |  PD={df_test["parkinsons_label"].sum()}, Healthy={(df_test["parkinsons_label"]==0).sum()}')
print(f'Saved train_ids.csv  -> {TRAIN_IDS_OUT}')
print(f'Saved test_ids.csv   -> {TEST_IDS_OUT}')
print()
print('>>> TEST SET IS NOW LOCKED. Do not inspect it for EDA, feature selection, or tuning.')

---
## 9. Train-Only Label-Aware EDA

**All analysis below uses `df_train` only.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ── Class distribution ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))
df_train['parkinsons_label'].value_counts().rename({0: 'Healthy (0)', 1: 'PD (1)'}).plot(
    kind='bar', ax=ax, color=['#4C9BE8', '#E8624C'], edgecolor='white'
)
ax.set_title('Train set — Label Distribution')
ax.set_xlabel('')
ax.set_ylabel('Participants')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(df_train['parkinsons_label'].value_counts().to_string())

In [ ]:
# ── Confound check: BirthYear and Gender by class (train only) ────────────────
# These are NOT in X but should be reported in limitations.

train_with_info = df_train[['participant_id', 'parkinsons_label']].merge(
    participant_info[['participant_id', 'birth_year', 'gender']],
    on='participant_id', how='left'
)
train_with_info['birth_year'] = pd.to_numeric(train_with_info['birth_year'], errors='coerce')

print('=== Confound Check: BirthYear by class (TRAIN only) ===')
print(train_with_info.groupby('parkinsons_label')['birth_year'].describe().to_string())
print()
print('=== Confound Check: Gender by class (TRAIN only) ===')
print(
    train_with_info.groupby(['parkinsons_label', 'gender'])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

In [ ]:
# ── Core timing features by class (TRAIN only) ────────────────────────────────
CORE_FEATURES = [
    'hold_mean', 'hold_std', 'hold_cv', 'hold_skew',
    'lat1_mean', 'lat1_std', 'lat1_cv',
    'lat2_mean', 'lat2_std', 'lat2_cv',
    'hold_LR_diff', 'hold_LR_ratio',
]
core_present = [c for c in CORE_FEATURES if c in df_train.columns]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(core_present[:12]):
    ax = axes[i]
    for label, colour in [(0, '#4C9BE8'), (1, '#E8624C')]:
        data = df_train.loc[df_train['parkinsons_label'] == label, col].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=colour,
                label='Healthy' if label == 0 else 'PD', edgecolor='none')
    ax.set_title(col, fontsize=8)
    ax.legend(fontsize=6)

for j in range(len(core_present), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Core Feature Distributions by Class (TRAIN only)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature null-rate report (train) ─────────────────────────────────────────
null_pct = df_train[feature_cols].isnull().mean().sort_values(ascending=False)
high_null = null_pct[null_pct > 0.10]
print(f'Features with > 10% null in train: {len(high_null)}')
if not high_null.empty:
    print(high_null.head(20).to_string())
    print()
    print('>>> These are sparse direction/hand sub-groups (e.g. lat1_SR). '
          'Consider dropping or imputing with median inside a Pipeline.')
else:
    print('No features with > 10% null.')

In [ ]:
print('=== Cleaning Pipeline Complete ===')
print(f'  features_clean.csv  : {FEATURES_OUT}')
print(f'  participant_info.csv: {PARTICIPANT_INFO}')
print(f'  train_ids.csv       : {TRAIN_IDS_OUT}')
print(f'  test_ids.csv        : {TEST_IDS_OUT}')
print()
print(f'  Final participants  : {len(features_clean)}')
print(f'  Feature columns     : {len(feature_cols)}')
print(f'  Train / Test        : {len(df_train)} / {len(df_test)}')